# División (Split) del dataset de perfiles completos

Con el fin de preparar los datos para el entrenamiento de modelos, en este notebook se genera una metadata persistente con una partición estratificada.

El objetivo es que otros notebooks puedan reutilizar exactamente la misma división `train` / `val` / `test` sin recalcularla.

El resultado se guarda como `metadata.csv` con una nueva columna `split`.

`CLASES_A_DIVIDIR` selecciona las clases que se procesan. Sus imágenes se copian a `DATA_DESTINO` conservando las carpetas de clase y las subcarpetas originales. La división `train` / `val` / `test` queda registrada en la metadata, sin crear carpetas por split. Al repetir la ejecución se sobrescriben los archivos coincidentes; no se eliminan archivos de ejecuciones anteriores.

In [1]:
# Importaciones
from vasijas.archivos import copiar_carpetas_de_clases
from vasijas.config import load_config
from vasijas.metadata import cargar_metadata, guardar_metadata
from vasijas.paths import ProjectPaths
from vasijas.split_dataset import (
    ConfiguracionSplitDataset,
    copiar_imagenes_split,
    realizar_split_en_metadata_perfiles_completos,
    describir_configuracion_split,
    advertir_imagenes_sin_metadata,
    mostrar_resumen_split,
    construir_resumen_split,
)

In [2]:
# Configuracion general
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

## Origen y destino de datos

In [3]:
# Origen y destino de datos
DATA_ORIGEN = PATHS.DATA_CUADRADOS
DATA_DESTINO = PATHS.DATA_DIVIDIDO

print(f"Dataset de imágenes a dividir: {DATA_ORIGEN}")
print(f"Destino de imágenes y archivo de metadata: {DATA_DESTINO}")

Dataset de imágenes a dividir: /home/rafael/Dev/vasijas/data/05.cuadrados
Destino de imágenes y archivo de metadata: /home/rafael/Dev/vasijas/data/10.cuadrados_divididos_en_train_val_y_test


## Copia de imágenes

A fin seguir el criterio de colocar en una misma carpeta las imágenes a procesar y el archivo metadata se copian las imágenes a la carperta de destino.

In [4]:
from vasijas.archivos import copiar_estructura_archivos

# Copiar toda la estructura de DATA_CORREGIDOS a DATA_UBICADOS
archivos_copiados = copiar_estructura_archivos(
    DATA_ORIGEN,
    DATA_DESTINO,
)

print(f"{archivos_copiados} archivos copiados.")
print(f"Desde {DATA_ORIGEN}\n" f"Hacia {DATA_DESTINO}")

1243 archivos copiados.
Desde /home/rafael/Dev/vasijas/data/05.cuadrados
Hacia /home/rafael/Dev/vasijas/data/10.cuadrados_divididos_en_train_val_y_test


## Vista previa al procesamiento

In [5]:
METADATA_ORIGEN = cargar_metadata(DATA_ORIGEN)

print("Muestra metadata origen:")
METADATA_ORIGEN.head()

Muestra metadata origen:


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla


## Configuración del split

In [6]:
# Parámetros del split
CLASES_A_DIVIDIR = [*range(1, 12)]

SEED = 42

# TODO MEJORAR, PASAR A UNA FUNCIÓN
config_split = ConfiguracionSplitDataset(semilla=SEED)

print(describir_configuracion_split(config_split))

Split configurado -> train: 70%, val: 15%, test: 15%, semilla: 42


## Construcción del split

Se usa `torchvision.datasets.ImageFolder` dentro del módulo para respetar el mismo orden de muestras y las mismas clases que consume el entrenamiento. `CLASES_A_DIVIDIR` permite limitar el split a un subconjunto de carpetas/clases. Luego se aplica `train_test_split` dos veces: primero `train` contra el conjunto temporal, y después `val` contra `test`.


In [7]:
resultado_split = realizar_split_en_metadata_perfiles_completos(
    data_origen=DATA_ORIGEN,
    metadata_origen=METADATA_ORIGEN,
    configuracion_split=config_split,
    clases_a_dividir=CLASES_A_DIVIDIR,
)

df_split = resultado_split.registros_split

mostrar_resumen_split(resultado_split)

Clases procesadas: ['clase01', 'clase02', 'clase03', 'clase04', 'clase05', 'clase06', 'clase07', 'clase08', 'clase09', 'clase10', 'clase11']
Imágenes procesadas: 1081
Split -> train: 756, val: 162, test: 163


In [8]:
# Tabla de archivos con split
df_split.head()

,Identificador,archivo,ruta_relativa,clase_carpeta,split
0,al_13b,al_13b.png,clase01/al_13b.png,clase01,train
1,al_14b,al_14b.png,clase01/al_14b.png,clase01,train
2,ba_115_4,ba_115_4.png,clase01/ba_115_4.png,clase01,train
3,ba_172_3,ba_172_3.png,clase01/ba_172_3.png,clase01,train
4,ba_191_2,ba_191_2.png,clase01/ba_191_2.png,clase01,train


## Unión con metadata

La metadata se limita a las imágenes procesadas y se agregan las columnas `archivo`, `ruta_relativa`, `clase_carpeta` y `split`.


In [9]:
df_metadata_split = resultado_split.metadata
advertir_imagenes_sin_metadata(resultado_split.ids_sin_metadata)
df_metadata_split.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,archivo,ruta_relativa,clase_carpeta,split
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla,al_03a.png,clase11/al_03a.png,clase11,test
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla,al_03b.png,clase11/al_03b.png,clase11,val
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla,al_03c.png,clase11/al_03c.png,clase11,val
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla,al_03d.png,clase11/al_03d.png,clase11,train
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla,al_03e.png,clase11/al_03e.png,clase11,val


In [10]:
# Control de distribución por split y clase
resumen_split = construir_resumen_split(df_metadata_split)
resumen_split

split,test,train,val,All
clase_carpeta,,,,
clase01,4,18,4,26
clase02,7,32,7,46
clase03,11,50,10,71
clase04,1,7,2,10
clase05,9,39,8,56
clase06,7,36,8,51
clase07,7,34,7,48
clase08,42,196,42,280
clase09,3,14,3,20


In [11]:
# Copia de imágenes seleccionadas y guardado de metadata
imagenes_copiadas = copiar_imagenes_split(
    data_origen=DATA_ORIGEN,
    data_destino=DATA_DESTINO,
    registros_split=df_split,
)

ruta_metadata_split = guardar_metadata(
    df=df_metadata_split,
    carpeta_destino=DATA_DESTINO,
)

print(f"Metadata con split guardada en: {ruta_metadata_split}")
print(f"Registros guardados: {len(df_metadata_split)}")
print(f"Imágenes copiadas: {imagenes_copiadas}")

Metadata con split guardada en: /home/rafael/Dev/vasijas/data/10.cuadrados_divididos_en_train_val_y_test/metadata.csv
Registros guardados: 1081
Imágenes copiadas: 1081


## Uso desde otros notebooks

```python
import pandas as pd

from vasijas.config import load_config
from vasijas.paths import ProjectPaths

CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

metadata_split = pd.read_csv(PATHS.DATA_DIVIDIDO / "metadata.csv")
train_df = metadata_split[metadata_split["split"] == "train"]
val_df = metadata_split[metadata_split["split"] == "val"]
test_df = metadata_split[metadata_split["split"] == "test"]
```
